# PointNet pour la classification de nuages de points

Ce notebook est une version découpée du script `pointnet.py`, organisée en étapes logiques avec de courtes explications.

## 1) Imports et dépendances

On charge les bibliothèques nécessaires pour le calcul, la lecture des fichiers PLY, PyTorch et les transforms.

In [ ]:
import numpy as np
import random
import math
import os
import time
from pathlib import Path
import importlib.util
import torch
import scipy.spatial.distance
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
import torch.nn as nn
import torch.nn.functional as F

def load_local_ply_module():
    """Charge le ply.py local du projet pour eviter le conflit avec le package pip 'ply'."""
    candidates = [
        Path.cwd() / 'ply.py',
        Path.cwd() / 'PointNetLab' / 'Code' / 'ply.py',
        Path.cwd() / 'Pointclouds-classification-with-the-POINTNET-Neural-network' / 'PointNetLab' / 'Code' / 'ply.py',
    ]
    for path in candidates:
        if path.exists():
            spec = importlib.util.spec_from_file_location('ply_local', path.resolve())
            ply_local = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(ply_local)
            return ply_local, path.resolve()
    raise FileNotFoundError('ply.py local introuvable. Place le notebook dans le repo ou ajuste les chemins candidats.')

ply_local, ply_path = load_local_ply_module()
write_ply = ply_local.write_ply
read_ply = ply_local.read_ply
print('PLY local charge depuis:', ply_path)

# Performance: enable cudnn benchmark for potential speedups
torch.backends.cudnn.benchmark = True


## 3) Dataset PyTorch (ModelNet en PLY)

Cette classe indexe les fichiers `.ply`, construit les couples `(nuage, label)` et applique les transformations.

In [ ]:
class PointCloudData(Dataset):
    """
    Dataset PyTorch pour ModelNet
    """

    def __init__(self,
                 root_dir,
                 folder="train",
                 transform=None):
        self.root_dir = root_dir

        folders = [dir for dir in sorted(os.listdir(root_dir))
                   if os.path.isdir(root_dir + "/" + dir)]

        self.classes = {folder: i for i, folder in enumerate(folders)}
        self.transforms = transform
        self.files = []

        for category in self.classes.keys():
            new_dir = root_dir + "/" + category + "/" + folder
            for file in os.listdir(new_dir):
                if file.endswith('.ply'):
                    sample = {}
                    sample['ply_path'] = new_dir + "/" + file
                    sample['category'] = category
                    self.files.append(sample)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        ply_path = self.files[idx]['ply_path']
        category = self.files[idx]['category']

        data = read_ply(ply_path)
        pointcloud = np.vstack((data['x'], data['y'], data['z'])).T.astype(np.float32)

        if self.transforms is not None:
            pointcloud = self.transforms(pointcloud)
        else:
            pointcloud = torch.from_numpy(pointcloud)

        label = self.classes[category]
        return {'pointcloud': pointcloud, 'category': label}


## 4) Modèle de base MLP

Un baseline simple qui aplatit le nuage puis applique des couches fully-connected.

In [ ]:
class PointMLP(nn.Module):
    def __init__(self, classes=40):
        super().__init__()
        self.flatten = nn.Flatten(start_dim=1)

        self.fc1 = nn.Linear(3072, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.act1 = nn.ReLU()

        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.act2 = nn.ReLU()
        self.drop = nn.Dropout(0.3)

        self.fc3 = nn.Linear(256, classes)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, input):
        x = self.flatten(input)

        x = self.fc1(x)
        x = self.bn1(x)
        x = self.act1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.drop(x)

        x = self.fc3(x)
        x = self.logsoftmax(x)
        return x

## 5) PointNet basique

Version convolutionnelle 1D sur les points, suivie d'un max pooling global et d'une tête de classification.

In [ ]:
class PointNetBasic(nn.Module):
    def __init__(self, classes=40):
        super().__init__()

        self.conv1 = nn.Conv1d(3, 64, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.act1 = nn.ReLU()

        self.conv2 = nn.Conv1d(64, 64, 1)
        self.bn2 = nn.BatchNorm1d(64)
        self.act2 = nn.ReLU()

        self.conv3 = nn.Conv1d(64, 64, 1)
        self.bn3 = nn.BatchNorm1d(64)
        self.act3 = nn.ReLU()

        self.conv4 = nn.Conv1d(64, 128, 1)
        self.bn4 = nn.BatchNorm1d(128)
        self.act4 = nn.ReLU()

        self.conv5 = nn.Conv1d(128, 1024, 1)
        self.bn5 = nn.BatchNorm1d(1024)
        self.act5 = nn.ReLU()

        self.maxpool5 = nn.MaxPool1d(1024)
        self.flatten = nn.Flatten(start_dim=1)

        self.fc6 = nn.Linear(1024, 512)
        self.bn6 = nn.BatchNorm1d(512)
        self.act6 = nn.ReLU()

        self.fc7 = nn.Linear(512, 256)
        self.bn7 = nn.BatchNorm1d(256)
        self.act7 = nn.ReLU()
        self.drop7 = nn.Dropout(0.3)

        self.fc8 = nn.Linear(256, classes)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, input):
        x = self.conv1(input)
        x = self.bn1(x)
        x = self.act1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.act3(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.act4(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = self.act5(x)

        x = self.maxpool5(x)
        x = self.flatten(x)

        x = self.fc6(x)
        x = self.bn6(x)
        x = self.act6(x)

        x = self.fc7(x)
        x = self.bn7(x)
        x = self.act7(x)
        x = self.drop7(x)

        x = self.fc8(x)
        x = self.logsoftmax(x)
        return x


## 6) T-Net et PointNet complet

Le T-Net apprend une transformation affine des points/features. Cette cellule reprend les classes du script original.

In [ ]:
class Tnet(nn.Module):
    def __init__(self, k=3):
        super().__init__()

        self.k = k

        self.conv1 = nn.Conv1d(k, 64, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.act1 = nn.ReLU()

        self.conv2 = nn.Conv1d(64, 128, 1)
        self.bn2 = nn.BatchNorm1d(128)
        self.act2 = nn.ReLU()

        self.conv3 = nn.Conv1d(128, 1024, 1)
        self.bn3 = nn.BatchNorm1d(1024)
        self.act3 = nn.ReLU()

        self.maxpool3 = nn.MaxPool1d(1024)
        self.flatten = nn.Flatten(start_dim=1)

        self.fc4 = nn.Linear(1024, 512)
        self.bn4 = nn.BatchNorm1d(512)
        self.act4 = nn.ReLU()

        self.fc5 = nn.Linear(512, 256)
        self.bn5 = nn.BatchNorm1d(256)
        self.act5 = nn.ReLU()

        self.fc6 = nn.Linear(256, k * k)

    def forward(self, input):
        x = self.conv1(input)
        x = self.bn1(x)
        x = self.act1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.act3(x)

        x = self.maxpool3(x)
        x = self.flatten(x)

        x = self.fc4(x)
        x = self.bn4(x)
        x = self.act4(x)

        x = self.fc5(x)
        x = self.bn5(x)
        x = self.act5(x)

        x = self.fc6(x)
        x = x.reshape(x.size(0), self.k, self.k)

        I = torch.eye(self.k, device=x.device, dtype=x.dtype)
        I = I.unsqueeze(0).repeat(x.size(0), 1, 1)
        x = x + I

        return x


class PointNetFull(nn.Module):
    # Version avec uniquement le premier T-NET (3x3), conforme à l'énoncé.
    def __init__(self, tnet1, classes=40):
        super().__init__()

        self.tnet1 = tnet1

        self.conv1 = nn.Conv1d(3, 64, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.act1 = nn.ReLU()

        self.conv2 = nn.Conv1d(64, 64, 1)
        self.bn2 = nn.BatchNorm1d(64)
        self.act2 = nn.ReLU()

        self.conv3 = nn.Conv1d(64, 64, 1)
        self.bn3 = nn.BatchNorm1d(64)
        self.act3 = nn.ReLU()

        self.conv4 = nn.Conv1d(64, 128, 1)
        self.bn4 = nn.BatchNorm1d(128)
        self.act4 = nn.ReLU()

        self.conv5 = nn.Conv1d(128, 1024, 1)
        self.bn5 = nn.BatchNorm1d(1024)
        self.act5 = nn.ReLU()

        self.maxpool5 = nn.MaxPool1d(1024)
        self.flatten = nn.Flatten(start_dim=1)

        self.fc6 = nn.Linear(1024, 512)
        self.bn6 = nn.BatchNorm1d(512)
        self.act6 = nn.ReLU()

        self.fc7 = nn.Linear(512, 256)
        self.bn7 = nn.BatchNorm1d(256)
        self.act7 = nn.ReLU()
        self.drop7 = nn.Dropout(0.3)

        self.fc8 = nn.Linear(256, classes)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, input):
        m3x3 = self.tnet1(input)
        x = torch.bmm(m3x3, input)

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.act1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.act2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.act3(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.act4(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = self.act5(x)

        x = self.maxpool5(x)
        x = self.flatten(x)

        x = self.fc6(x)
        x = self.bn6(x)
        x = self.act6(x)

        x = self.fc7(x)
        x = self.bn7(x)
        x = self.act7(x)
        x = self.drop7(x)

        x = self.fc8(x)
        x = self.logsoftmax(x)

        return x, m3x3


## 7) Fonctions de perte

`basic_loss` utilise NLLLoss. `pointnet_full_loss` ajoute une régularisation d'orthogonalité pour le T-Net.

In [ ]:
def basic_loss(outputs, labels):
    criterion = torch.nn.NLLLoss()
    return criterion(outputs, labels)


def pointnet_full_loss(outputs, labels, m3x3, alpha=0.001):
    criterion = torch.nn.NLLLoss()
    bsize = outputs.size(0)

    id3x3 = torch.eye(3, device=outputs.device, dtype=outputs.dtype).unsqueeze(0).repeat(bsize, 1, 1)
    diff3x3 = id3x3 - torch.bmm(m3x3, m3x3.transpose(1, 2))

    return criterion(outputs, labels) + alpha * torch.norm(diff3x3) / float(bsize)


## 8) Boucle d'entraînement

La fonction `train` gère l'optimiseur, le scheduler, l'entraînement batch par batch et le calcul d'accuracy sur le test.

In [ ]:
def train(model, device, train_loader, test_loader=None, epochs=250):
    from torch.cuda.amp import autocast, GradScaler
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    amp_enabled = (device.type == 'cuda')
    scaler = GradScaler(enabled=amp_enabled)

    for epoch in range(epochs):
        model.train()
        t0 = time.perf_counter()

        for i, data in enumerate(train_loader, 0):
            inputs = data['pointcloud'].to(device, non_blocking=True).float()
            labels = data['category'].to(device, non_blocking=True)

            optimizer.zero_grad()
            with autocast(enabled=amp_enabled):
                model_out = model(inputs.transpose(1, 2))
                if isinstance(model_out, tuple):
                    outputs, m3x3 = model_out
                    loss = pointnet_full_loss(outputs, labels, m3x3)
                else:
                    outputs = model_out
                    loss = basic_loss(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        if device.type == 'cuda':
            torch.cuda.synchronize()
        epoch_time = time.perf_counter() - t0

        val_acc = None
        if test_loader:
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for data in test_loader:
                    inputs = data['pointcloud'].to(device, non_blocking=True).float()
                    labels = data['category'].to(device, non_blocking=True)
                    with autocast(enabled=amp_enabled):
                        model_out = model(inputs.transpose(1, 2))
                        outputs = model_out[0] if isinstance(model_out, tuple) else model_out
                    _, predicted = torch.max(outputs, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            val_acc = 100. * correct / total
            print(f'Epoch: {epoch+1}, Time: {epoch_time:.2f}s, Loss: {loss:.3f}, Test accuracy: {val_acc:.1f} %')
        else:
            print(f'Epoch: {epoch+1}, Time: {epoch_time:.2f}s, Loss: {loss:.3f}')

        scheduler.step()


## 9) Chargement du dataset et DataLoaders

On instancie les jeux d'entraînement/test puis on affiche quelques infos de contrôle.

In [ ]:
DATA_ROOT = "/home/nochi/NOCHI/M2_PAR/Apprenyissage_Pointcloud/PointNetLab/data/ModelNet10_PLY"

# Creation brute des datasets (sans augmentation a ce stade).
train_ds = PointCloudData(DATA_ROOT, folder='train')
test_ds = PointCloudData(DATA_ROOT, folder='test')

inv_classes = {i: cat for cat, i in train_ds.classes.items()}
print('Classes:', inv_classes)
print('Train dataset size:', len(train_ds))
print('Test dataset size:', len(test_ds))
print('Number of classes:', len(train_ds.classes))
print('Sample pointcloud shape (sans augmentation):', train_ds[0]['pointcloud'].size())


## 9 bis) Transformations et data augmentation

Fonctions d'augmentation appliquees au dataset d'entrainement apres creation des datasets.


In [ ]:
class RandomRotation_z(object):
    # Rotation autour de l'axe Z pour rendre le modele moins sensible a l'orientation horizontale.
    def __call__(self, pointcloud):
        theta = random.random() * 2. * math.pi
        rot_matrix = np.array([[math.cos(theta), -math.sin(theta),      0],
                               [math.sin(theta),  math.cos(theta),      0],
                               [0,                              0,      1]])
        rot_pointcloud = rot_matrix.dot(pointcloud.T).T
        return rot_pointcloud


class RandomNoise(object):
    # Ajout de bruit gaussien faible pour ameliorer la robustesse aux perturbations capteur.
    def __call__(self, pointcloud):
        noise = np.random.normal(0, 0.02, (pointcloud.shape))
        noisy_pointcloud = pointcloud + noise
        return noisy_pointcloud


class ShufflePoints(object):
    # Melange l'ordre des points pour forcer l'invariance a la permutation.
    def __call__(self, pointcloud):
        np.random.shuffle(pointcloud)
        return pointcloud


class ToTensor(object):
    # Conversion finale numpy -> torch tensor.
    def __call__(self, pointcloud):
        return torch.from_numpy(pointcloud)


def default_transforms():
    # Pipeline historique du notebook.
    return transforms.Compose([
        RandomRotation_z(),
        RandomNoise(),
        ShufflePoints(),
        ToTensor()
    ])


## 9 ter) Augmentations supplementaires (mode manuel)

Presets d'augmentation (`none`, `moderate`, `strong`) et application aux DataLoaders.


In [ ]:
class RandomScale(object):
    # Mise a l'echelle isotrope aleatoire pour gerer des variations de taille globale.
    def __init__(self, scale_low=0.8, scale_high=1.25):
        self.scale_low = scale_low
        self.scale_high = scale_high

    def __call__(self, pointcloud):
        scale = np.random.uniform(self.scale_low, self.scale_high)
        return pointcloud * scale


class RandomTranslation(object):
    # Translation aleatoire courte pour limiter la dependance a la position absolue.
    def __init__(self, shift_range=0.1):
        self.shift_range = shift_range

    def __call__(self, pointcloud):
        shift = np.random.uniform(-self.shift_range, self.shift_range, (1, 3))
        return pointcloud + shift


class RandomPointDropout(object):
    # Supprime aleatoirement une fraction de points pour simuler des occultations.
    def __init__(self, max_dropout_ratio=0.2):
        self.max_dropout_ratio = max_dropout_ratio

    def __call__(self, pointcloud):
        dropout_ratio = np.random.uniform(0.0, self.max_dropout_ratio)
        drop_mask = np.random.random(pointcloud.shape[0]) <= dropout_ratio
        if np.any(drop_mask):
            pointcloud[drop_mask, :] = pointcloud[0, :]
        return pointcloud


class RandomMirrorXY(object):
    # Symetrie aleatoire selon X/Y pour augmenter la variabilite geometrique.
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, pointcloud):
        if np.random.rand() < self.p:
            pointcloud[:, 0] *= -1
        if np.random.rand() < self.p:
            pointcloud[:, 1] *= -1
        return pointcloud


def no_augmentation_transforms():
    # Dataset de test: pas d'augmentation, uniquement conversion tensor.
    return transforms.Compose([
        ToTensor()
    ])


def moderate_transforms():
    # Preset modere pour entrainement standard.
    return transforms.Compose([
        RandomRotation_z(),
        RandomScale(0.9, 1.1),
        RandomTranslation(0.05),
        RandomNoise(),
        ShufflePoints(),
        ToTensor()
    ])


def strong_transforms():
    # Preset fort pour pousser la robustesse (plus agressif).
    return transforms.Compose([
        RandomRotation_z(),
        RandomScale(0.75, 1.25),
        RandomTranslation(0.12),
        RandomMirrorXY(p=0.5),
        RandomNoise(),
        RandomPointDropout(max_dropout_ratio=0.2),
        ShufflePoints(),
        ToTensor()
    ])


AUGMENTATION_PRESET = 'moderate'  # options: 'none', 'moderate', 'strong'


def get_train_transforms(preset=AUGMENTATION_PRESET):
    presets = {
        'none': no_augmentation_transforms,
        'moderate': moderate_transforms,
        'strong': strong_transforms,
    }
    if preset not in presets:
        raise ValueError(f"Preset inconnu: {preset}. Choisir parmi {list(presets.keys())}")
    return presets[preset]()


# Application des augmentations apres creation des datasets.
train_ds.transforms = get_train_transforms(AUGMENTATION_PRESET)
test_ds.transforms = no_augmentation_transforms()
print("Preset d'augmentation courant (train):", AUGMENTATION_PRESET)
print('Sample pointcloud shape apres transforms:', train_ds[0]['pointcloud'].size())

# DataLoader: use multiple workers and pin_memory to reduce CPU->GPU overhead
train_loader = DataLoader(dataset=train_ds, batch_size=32, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(dataset=test_ds, batch_size=32,
                         num_workers=4, pin_memory=True, persistent_workers=True)


## 10) Visualisation d'un point cloud aléatoire

Affiche un échantillon du `train_ds` choisi au hasard avec son label.


In [ ]:
import matplotlib.pyplot as plt

def show_random_pointcloud(dataset, inv_classes, title_prefix='VISUALISATION RANDOM TRAIN SAMPLE'):
    idx = random.randint(0, len(dataset) - 1)
    sample = dataset[idx]
    points = sample['pointcloud']

    if torch.is_tensor(points):
        points = points.numpy()

    label_id = sample['category']
    label_name = inv_classes[label_id]

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], s=2, c=points[:, 2], cmap='viridis')
    ax.set_title(f'{title_prefix} | idx={idx} | classe={label_name}')
    ax.set_axis_off()
    plt.show()


show_random_pointcloud(train_ds, inv_classes)


## 10 bis) Visualisations supplementaires des augmentations

Comparaison directe du meme nuage de points avant/apres pipelines d'augmentation.


In [ ]:
def to_numpy_points(pc):
    if torch.is_tensor(pc):
        return pc.detach().cpu().numpy()
    return np.asarray(pc)


def read_raw_points(dataset, idx):
    ply_path = dataset.files[idx]['ply_path']
    data = read_ply(ply_path)
    pts = np.vstack((data['x'], data['y'], data['z'])).T
    return pts.astype(np.float32), ply_path


def plot_cloud(ax, pts, title):
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2, c=pts[:, 2], cmap='viridis')
    ax.set_title(title)
    ax.set_axis_off()


idx = random.randint(0, len(train_ds) - 1)
raw_points, raw_path = read_raw_points(train_ds, idx)
label_id = train_ds[idx]['category']
label_name = inv_classes[label_id]

pipelines = {
    'original': None,
    'default': default_transforms(),
    'moderate': moderate_transforms(),
    'strong': strong_transforms(),
}

fig = plt.figure(figsize=(18, 4))
for i, (name, trf) in enumerate(pipelines.items(), start=1):
    pts = raw_points.copy() if trf is None else to_numpy_points(trf(raw_points.copy()))
    ax = fig.add_subplot(1, len(pipelines), i, projection='3d')
    plot_cloud(ax, pts, name)

fig.suptitle(f'Comparaison augmentations | classe={label_name} | idx={idx}')
plt.tight_layout()
plt.show()
print('Fichier source:', raw_path)



## 10 ter) Projections 2D et distribution radiale

Permet de verifier visuellement l'impact geometrique des augmentations fortes.


In [ ]:
idx = random.randint(0, len(train_ds) - 1)
raw_points, _ = read_raw_points(train_ds, idx)
aug_points = to_numpy_points(strong_transforms()(raw_points.copy()))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0, 0].scatter(raw_points[:, 0], raw_points[:, 1], s=2, c=raw_points[:, 2], cmap='viridis')
axes[0, 0].set_title('Original XY')
axes[0, 1].scatter(raw_points[:, 0], raw_points[:, 2], s=2, c=raw_points[:, 1], cmap='viridis')
axes[0, 1].set_title('Original XZ')
axes[0, 2].scatter(raw_points[:, 1], raw_points[:, 2], s=2, c=raw_points[:, 0], cmap='viridis')
axes[0, 2].set_title('Original YZ')

axes[1, 0].scatter(aug_points[:, 0], aug_points[:, 1], s=2, c=aug_points[:, 2], cmap='magma')
axes[1, 0].set_title('Strong aug XY')
axes[1, 1].scatter(aug_points[:, 0], aug_points[:, 2], s=2, c=aug_points[:, 1], cmap='magma')
axes[1, 1].set_title('Strong aug XZ')
axes[1, 2].scatter(aug_points[:, 1], aug_points[:, 2], s=2, c=aug_points[:, 0], cmap='magma')
axes[1, 2].set_title('Strong aug YZ')

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

raw_r = np.linalg.norm(raw_points - raw_points.mean(axis=0, keepdims=True), axis=1)
aug_r = np.linalg.norm(aug_points - aug_points.mean(axis=0, keepdims=True), axis=1)

plt.figure(figsize=(7, 4))
plt.hist(raw_r, bins=40, alpha=0.6, label='original')
plt.hist(aug_r, bins=40, alpha=0.6, label='strong aug')
plt.title('Distribution des distances au centroide')
plt.xlabel('Distance')
plt.ylabel('Frequence')
plt.legend()
plt.show()



## 11) Instanciation du modèle et configuration device

On choisit l'architecture, on compte les paramètres entraînables et on déplace le modèle sur CPU/GPU.

In [ ]:
Tnet1 = Tnet(k=3)

# model = PointMLP(classes=10)
model = PointNetBasic(classes=10)
# model = PointNetFull(classes=10, tnet1=Tnet1)

model_parameters = filter(lambda p: p.requires_grad, model.parameters())
print('Number of parameters in the Neural Networks:', sum([np.prod(p.size()) for p in model_parameters]))

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model.to(device)


## 12) Lancement de l'entraînement

Cette cellule démarre l'entraînement complet et affiche le temps total.

In [ ]:
t0 = time.time()
train(model, device, train_loader, test_loader, epochs=20)
print('Total time for training :', time.time() - t0)